# Colab GPU Benchmark

This notebook installs the project in editable mode, confirms GPU access, runs the learned-model comparison on Colab, saves checkpoints, and exports the resulting artifacts.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = 'https://github.com/speck-christian/event_simulator.git'
MOUNT_DRIVE = False  # Set True if you want outputs copied to Google Drive\n
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/event_simulator_runs'\n

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/event_simulator')
if not REPO_DIR.exists():
    if not REPO_URL:
        raise FileNotFoundError('Set REPO_URL in the previous cell or clone the repo manually into /content/event_simulator.')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Working directory:', REPO_DIR)

In [ ]:
from pathlib import Path

DRIVE_OUTPUT_DIR = None
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR = Path(DRIVE_OUTPUT_ROOT) / 'adaptive_richer_60_20_rebalanced_learned'
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print('Drive output directory:', DRIVE_OUTPUT_DIR)
else:
    print('Drive mount disabled')

In [ ]:
!pip install -e .[colab]

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
OUTPUT_DIR = Path('analysis/adaptive_richer_60_20_rebalanced_learned')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
!python train_learned.py --device cuda --control-mode adaptive --simulation-profile richer --train-runs 60 --eval-runs 20 --duration 300 --epochs 14 --cache-dir analysis/cache_adaptive_richer_60_20_rebalanced --output-dir {OUTPUT_DIR} --checkpoint-dir {CHECKPOINT_DIR}

In [ ]:
from IPython.display import IFrame
IFrame('analysis/adaptive_richer_60_20_rebalanced_learned/model_comparison.html', width=1400, height=2200)

In [ ]:
import json
from pathlib import Path

runtime_path = Path('analysis/adaptive_richer_60_20_rebalanced_learned/runtime_summary.json')
runtime = json.loads(runtime_path.read_text())
print(json.dumps(runtime, indent=2))

In [ ]:
!cd analysis && zip -r adaptive_richer_60_20_rebalanced_learned_artifacts.zip adaptive_richer_60_20_rebalanced_learned

In [ ]:
import shutil
from pathlib import Path

if DRIVE_OUTPUT_DIR is not None:
    src = Path('analysis/adaptive_richer_60_20_rebalanced_learned')
    if DRIVE_OUTPUT_DIR.exists():
        shutil.rmtree(DRIVE_OUTPUT_DIR)
    shutil.copytree(src, DRIVE_OUTPUT_DIR)
    shutil.copy2('analysis/adaptive_richer_60_20_rebalanced_learned_artifacts.zip', DRIVE_OUTPUT_DIR.parent / 'adaptive_richer_60_20_rebalanced_learned_artifacts.zip')
    print('Copied artifacts to', DRIVE_OUTPUT_DIR)
else:
    print('Drive copy skipped')

In [ ]:
from google.colab import files
files.download('analysis/adaptive_richer_60_20_rebalanced_learned_artifacts.zip')

In [ ]:
import json
from pathlib import Path

LOCAL_RUNTIME_SUMMARY = Path('/content/runtime_local_summary.json')  # Upload a local runtime_summary.json here to compare
COLAB_RUNTIME_SUMMARY = Path('analysis/adaptive_richer_60_20_rebalanced_learned/runtime_summary.json')

if LOCAL_RUNTIME_SUMMARY.exists() and COLAB_RUNTIME_SUMMARY.exists():
    local = json.loads(LOCAL_RUNTIME_SUMMARY.read_text())
    colab = json.loads(COLAB_RUNTIME_SUMMARY.read_text())
    print('Overall speedup:', round(local['total_wall_seconds'] / colab['total_wall_seconds'], 3), 'x')
    for name, stats in colab['models'].items():
        if name in local['models']:
            fit_speedup = local['models'][name]['fit_seconds'] / max(1e-9, stats['fit_seconds'])
            eval_speedup = local['models'][name]['eval_seconds'] / max(1e-9, stats['eval_seconds'])
            total_speedup = local['models'][name]['total_seconds'] / max(1e-9, stats['total_seconds'])
            print(name, {'fit_speedup_x': round(fit_speedup, 3), 'eval_speedup_x': round(eval_speedup, 3), 'total_speedup_x': round(total_speedup, 3)})
else:
    print('To compare runtime here, upload a local runtime_summary.json to /content/runtime_local_summary.json first.')

## Optional full benchmark

Run the command below if you want the learned models on GPU but still want the simple baselines and mechanistic model in the same report.

```bash
python compare_models.py --device cuda --control-mode adaptive --simulation-profile richer --train-runs 60 --eval-runs 20 --duration 300 --epochs 14 --cache-dir analysis/cache_adaptive_richer_60_20_rebalanced --output-dir analysis/adaptive_richer_60_20_rebalanced_full
```